In [1]:
# -*- coding: utf-8 -*-
"""
1D Baseline: BL-PINN (Joint Training Paradigm)
- Adapted directly from the 3-Module Hard-Constraint FL-DAE architecture.
- Trains all 3 neural network modules simultaneously using a single global composite loss.
- Total iterations matches FL-DAE total budget (20,000 steps).
- End-to-end backpropagation (NO .detach() blocking between components).
- Full statistical reporting (Mean \pm Sample Std) and curves saved automatically.
- Point bookkeeping accurately corrected to 2*N_M for the matching condition.
"""

import time
import os
import random
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.optim as optim
from scipy.stats import qmc
from scipy.spatial import cKDTree

# =============================================================================
# Basic settings
# =============================================================================
Tensor = torch.Tensor
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = False

SEEDS = [33, 99, 202, 1234, 5678, 9999]
MU_LIST = [0.01, 0.001, 0.0001]

X_MIN, X_MAX, T_FINAL = 0.0, 1.0, 0.3
L_VALUE, R_VALUE, H0_VALUE = -10.0, 5.0, 0.1

DEPTH, WIDTH = 4, 10
LR = 1.0e-3
XI_MAX = 12.0

# BL-PINN 联合训练总步数对齐 FL-DAE
TOTAL_JOINT_ITERS = 20000

N_PHI = 1000
N_H = 1000
N_Q = 2000       # 每侧内部残差采样点数
N_M = 2000       # 界面匹配采样点数

# 【精准点数统计修正】匹配项严格计入 2*N_M （左侧 N_M + 右侧 N_M）
TOTAL_LOSS_POINT_STEPS = TOTAL_JOINT_ITERS * (N_PHI + N_H + 2 * N_Q + 2 * N_M)
TOTAL_EFFECTIVE_EVAL_STEPS = TOTAL_JOINT_ITERS * (2 * N_PHI + N_H + 2 * N_Q + 2 * N_M)

AVERAGE_LOSS_POINTS_PER_STEP = TOTAL_LOSS_POINT_STEPS / TOTAL_JOINT_ITERS
AVERAGE_EFFECTIVE_EVALS_PER_STEP = TOTAL_EFFECTIVE_EVAL_STEPS / TOTAL_JOINT_ITERS

NUM_SAMPLES = 5000
LHS_SEED = 1234
EVAL_WARMUP = 20
EVAL_REPEAT = 200
SAVE_FULL_GRID = False
NX_FULL, NT_FULL = 201, 201
BASE_PATH = "."

# =============================================================================
# Utilities
# =============================================================================
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def grad(outputs: Tensor, inputs: Tensor, create_graph: bool = True, retain_graph: bool = True) -> Tensor:
    return torch.autograd.grad(outputs, inputs, grad_outputs=torch.ones_like(outputs),
                               create_graph=create_graph, retain_graph=retain_graph, only_inputs=True)[0]

def mse(x: Tensor) -> Tensor: return torch.mean(x ** 2)
def source_f(x: Tensor) -> Tensor: return x - x ** 2 + x ** 3
def mean_std(values):
    arr = np.asarray(values, dtype=float)
    return (np.nanmean(arr), 0.0) if len(arr) <= 1 else (np.nanmean(arr), np.nanstd(arr, ddof=1))

# =============================================================================
# Neural networks (3 Modules with Multi-Head Outputs)
# =============================================================================
class MLP(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, width: int, depth: int):
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(in_dim, width), nn.Tanh()]
        for _ in range(depth - 2): layers += [nn.Linear(width, width), nn.Tanh()]
        layers.append(nn.Linear(width, out_dim))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.net:
            if isinstance(module, nn.Linear):
                init.xavier_normal_(module.weight)
                if module.bias is not None: init.zeros_(module.bias)

    def forward(self, x: Tensor) -> Tensor: return self.net(x)

class BLPINN_3Module(nn.Module):
    def __init__(self):
        super().__init__()
        self.N_phi = MLP(1, 2, WIDTH, DEPTH) 
        self.N_h = MLP(1, 1, WIDTH, DEPTH)
        self.N_Q = MLP(3, 2, WIDTH, DEPTH)   

    def phi_m(self, x: Tensor) -> Tensor:
        return L_VALUE + (x - X_MIN) * self.N_phi(x)[:, 0:1]

    def phi_p(self, x: Tensor) -> Tensor:
        return R_VALUE + (x - X_MAX) * self.N_phi(x)[:, 1:2]

    def h(self, t: Tensor) -> Tensor:
        return H0_VALUE + t * self.N_h(t)

    def Q_m(self, xi: Tensor, h: Tensor, t: Tensor) -> Tensor:
        raw = self.N_Q(torch.cat([xi, h, t], dim=1))[:, 0:1]
        return ((xi + XI_MAX) / XI_MAX) * raw

    def Q_p(self, xi: Tensor, h: Tensor, t: Tensor) -> Tensor:
        raw = self.N_Q(torch.cat([xi, h, t], dim=1))[:, 1:2]
        return ((XI_MAX - xi) / XI_MAX) * raw

# =============================================================================
# Static sampling (确保这些函数在循环前被正确加载)
# =============================================================================
def sample_x(n: int, requires_grad: bool = False) -> Tensor:
    x = X_MIN + (X_MAX - X_MIN) * torch.rand(n, 1, device=DEVICE)
    x.requires_grad_(requires_grad)
    return x

def sample_t(n: int, requires_grad: bool = False) -> Tensor:
    t = T_FINAL * torch.rand(n, 1, device=DEVICE)
    t.requires_grad_(requires_grad)
    return t

def sample_xi_minus(n: int) -> Tensor:
    xi = -XI_MAX * torch.rand(n, 1, device=DEVICE)
    xi.requires_grad_(True)
    return xi

def sample_xi_plus(n: int) -> Tensor:
    xi = XI_MAX * torch.rand(n, 1, device=DEVICE)
    xi.requires_grad_(True)
    return xi

# =============================================================================
# Losses (打通计算图，允许梯度回传)
# =============================================================================
def loss_outer_phi(model: BLPINN_3Module, x_phi: Tensor) -> Tensor:
    pm = model.phi_m(x_phi)
    pp = model.phi_p(x_phi)
    dpm = grad(pm, x_phi, create_graph=True, retain_graph=True)
    dpp = grad(pp, x_phi, create_graph=True, retain_graph=True)
    ff = source_f(x_phi)
    return mse(pm * dpm - ff) + mse(pp * dpp - ff)

def loss_interface_h(model: BLPINN_3Module, t_h: Tensor) -> Tensor:
    h = model.h(t_h)
    h_t = grad(h, t_h, create_graph=True, retain_graph=True)
    pm_h = model.phi_m(h)
    pp_h = model.phi_p(h)
    return mse(h_t + 0.5 * (pm_h + pp_h))

def q_residual_side_m(model: BLPINN_3Module, xi_base: Tensor, t_base: Tensor) -> Tensor:
    t = t_base.clone().requires_grad_(True)
    h_val = model.h(t)
    h_t = grad(h_val, t, create_graph=True, retain_graph=True)
    phi_side = model.phi_m(h_val)

    xi = xi_base.clone().requires_grad_(True)
    Qs = model.Q_m(xi, h_val, t)
    Q_xi = grad(Qs, xi, create_graph=True, retain_graph=True)
    Q_xixi = grad(Q_xi, xi, create_graph=True, retain_graph=True)
    return Q_xixi + (h_t + phi_side + Qs) * Q_xi

def q_residual_side_p(model: BLPINN_3Module, xi_base: Tensor, t_base: Tensor) -> Tensor:
    t = t_base.clone().requires_grad_(True)
    h_val = model.h(t)
    h_t = grad(h_val, t, create_graph=True, retain_graph=True)
    phi_side = model.phi_p(h_val)

    xi = xi_base.clone().requires_grad_(True)
    Qs = model.Q_p(xi, h_val, t)
    Q_xi = grad(Qs, xi, create_graph=True, retain_graph=True)
    Q_xixi = grad(Q_xi, xi, create_graph=True, retain_graph=True)
    return Q_xixi + (h_t + phi_side + Qs) * Q_xi

def loss_inner_Q(model: BLPINN_3Module, xi_m: Tensor, t_m: Tensor, xi_p: Tensor, t_p: Tensor, t_match: Tensor) -> Tensor:
    res_m = q_residual_side_m(model, xi_m, t_m)
    res_p = q_residual_side_p(model, xi_p, t_p)
    loss_res = mse(res_m) + mse(res_p)

    h_match = model.h(t_match)
    pm_h = model.phi_m(h_match)
    pp_h = model.phi_p(h_match)
    phi_mid = 0.5 * (pm_h + pp_h)

    xi0 = torch.zeros_like(t_match)
    Q0_m = model.Q_m(xi0, h_match, t_match)
    Q0_p = model.Q_p(xi0, h_match, t_match)
    loss_match = mse(pm_h + Q0_m - phi_mid) + mse(pp_h + Q0_p - phi_mid)

    return loss_res + loss_match

# =============================================================================
# Unified Joint Training Loop (Zero-overhead appending aligned with DAE)
# =============================================================================
def train_joint(model: BLPINN_3Module, x_phi, t_h, xi_m, t_m, xi_p, t_p, t_match) -> Tuple[float, List[Tensor]]:
    for p in model.parameters(): p.requires_grad_(True)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    loss_list = []
    
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()
    model.train()
    
    for _ in range(TOTAL_JOINT_ITERS):
        optimizer.zero_grad(set_to_none=True)
        
        l_phi = loss_outer_phi(model, x_phi)
        l_h = loss_interface_h(model, t_h)
        l_Q = loss_inner_Q(model, xi_m, t_m, xi_p, t_p, t_match)
        
        loss = l_phi + l_h + l_Q
        loss.backward()
        optimizer.step()
        
        # 仅追加计算图解耦指针，避免任何每步同步开销
        loss_list.append(loss.detach())
        
    if torch.cuda.is_available(): torch.cuda.synchronize()
    return time.perf_counter() - t0, loss_list

# =============================================================================
# Reconstruction
# =============================================================================
def reconstruct_bipinn_gpu(model: BLPINN_3Module, x_eval: Tensor, t_eval: Tensor, mu: float) -> Tensor:
    with torch.no_grad():
        h = model.h(t_eval)
        xi = (x_eval - h) / mu
        xi_l = torch.clamp(xi, min=-XI_MAX, max=0.0)
        xi_r = torch.clamp(xi, min=0.0, max=XI_MAX)
        phi_m_val = model.phi_m(x_eval)
        phi_p_val = model.phi_p(x_eval)
        Q_l = model.Q_m(xi_l, h, t_eval)
        Q_r = model.Q_p(xi_r, h, t_eval)
        mask = (x_eval <= h).to(x_eval.dtype)
        return mask * (phi_m_val + Q_l) + (1.0 - mask) * (phi_p_val + Q_r)

# =============================================================================
# LHS test setup
# =============================================================================
def true_u0_filename(mu): return f"1d_U0_true_mu{mu:.0e}_201_201_Mathematica.csv"
def lhs_index_filename(mu): return f"1d_LHS_sample_indices_mu{mu:.0e}.npy"
def lhs_points_filename(mu): return f"1d_LHS_test_points_mu{mu:.0e}.csv"

def load_true_solution_u0(mu):
    filename = true_u0_filename(mu)
    path = os.path.join(BASE_PATH, filename)
    if not os.path.exists(path): raise FileNotFoundError(f"Missing: {filename}")
    df = pd.read_csv(path).sort_values(by=["t", "x"]).reset_index(drop=True)
    return df, filename

def generate_lhs_indices(df_true, mu):
    total_points = len(df_true)
    t_min, t_max = df_true["t"].min(), df_true["t"].max()
    x_min, x_max = df_true["x"].min(), df_true["x"].max()
    all_points = df_true[["t", "x"]].values
    kdtree = cKDTree(all_points)
    selected, used, batch_id = [], set(), 0
    while len(selected) < NUM_SAMPLES and batch_id < 100:
        sampler = qmc.LatinHypercube(d=2, seed=LHS_SEED + batch_id)
        lhs_sample_scaled = qmc.scale(sampler.random(n=NUM_SAMPLES), [t_min, x_min], [t_max, x_max])
        _, candidate_indices = kdtree.query(lhs_sample_scaled)
        for idx in candidate_indices:
            idx = int(idx)
            if idx not in used:
                used.add(idx)
                selected.append(idx)
                if len(selected) == NUM_SAMPLES: break
        batch_id += 1
    if len(selected) < NUM_SAMPLES:
        remaining = np.setdiff1d(np.arange(total_points), np.asarray(selected, dtype=int), assume_unique=False)
        fill = np.random.default_rng(LHS_SEED).choice(remaining, size=NUM_SAMPLES - len(selected), replace=False)
        selected.extend([int(i) for i in fill])
    return np.asarray(selected, dtype=int)

def build_or_load_lhs_test_set_from_true(mu):
    df_true, filename = load_true_solution_u0(mu)
    index_file = lhs_index_filename(mu)
    if os.path.exists(index_file):
        sample_indices = np.load(index_file)
        valid = (len(sample_indices) == NUM_SAMPLES and len(np.unique(sample_indices)) == NUM_SAMPLES and np.max(sample_indices) < len(df_true))
        if not valid: sample_indices = generate_lhs_indices(df_true, mu)
    else:
        sample_indices = generate_lhs_indices(df_true, mu)
    np.save(index_file, sample_indices)
    
    t_lhs_np = df_true.iloc[sample_indices]["t"].values.reshape(-1, 1)
    x_lhs_np = df_true.iloc[sample_indices]["x"].values.reshape(-1, 1)
    true_lhs_np = df_true.iloc[sample_indices].iloc[:, 2].values.reshape(-1)
    
    df_test = pd.DataFrame({"t": t_lhs_np.reshape(-1), "x": x_lhs_np.reshape(-1), "u_true": true_lhs_np})
    df_test.to_csv(lhs_points_filename(mu), index=False)
    
    return {
        "x_lhs": torch.tensor(x_lhs_np, dtype=torch.float32, device=DEVICE),
        "t_lhs": torch.tensor(t_lhs_np, dtype=torch.float32, device=DEVICE),
        "t_lhs_np": t_lhs_np, "x_lhs_np": x_lhs_np, "true_lhs_np": true_lhs_np,
        "n_test": len(sample_indices)
    }

def compute_error(true_u, pred_u):
    diff = pred_u - true_u
    return np.linalg.norm(diff) / np.linalg.norm(true_u), np.max(np.abs(diff))

def save_full_grid_prediction(model: BLPINN_3Module, mu, seed):
    x_vals, t_vals = np.linspace(X_MIN, X_MAX, NX_FULL), np.linspace(0.0, T_FINAL, NT_FULL)
    T_grid, X_grid = np.meshgrid(t_vals, x_vals, indexing="ij")
    x_eval = torch.tensor(X_grid.reshape(-1, 1), dtype=torch.float32, device=DEVICE)
    t_eval = torch.tensor(T_grid.reshape(-1, 1), dtype=torch.float32, device=DEVICE)
    u_pred = reconstruct_bipinn_gpu(model, x_eval, t_eval, mu).detach().cpu().numpy().reshape(-1)
    pd.DataFrame({"x": X_grid.reshape(-1), "t": T_grid.reshape(-1), "u": u_pred}).to_csv(f"1d_BLPINN_U0_predicted_full_mu{mu:.0e}_seed{seed}.csv", index=False)

# =============================================================================
# Main execution flow
# =============================================================================
if __name__ == "__main__":
    print("\n" + "=" * 80)
    print("Starting 1D BL-PINN benchmark with strict LHS-based T_eval and error")
    print("=" * 80 + "\n")
    print(f"Device: {DEVICE}")
    print(f"LHS test points: N_test={NUM_SAMPLES}, LHS seed={LHS_SEED}")
    print(f"T_eval warmup: {EVAL_WARMUP}, repeated timing: {EVAL_REPEAT}\n")

    lhs_data = {mu: build_or_load_lhs_test_set_from_true(mu) for mu in MU_LIST}
    metrics = {mu: [] for mu in MU_LIST}

    for seed in SEEDS:
        print("\n" + "-" * 80)
        print(f"Running seed = {seed}")
        print("-" * 80)
        set_seed(seed)

        x_phi = sample_x(N_PHI, requires_grad=True)
        t_h = sample_t(N_H, requires_grad=True)
        xi_m = sample_xi_minus(N_Q)
        t_m = sample_t(N_Q, requires_grad=False)
        xi_p = sample_xi_plus(N_Q)
        t_p = sample_t(N_Q, requires_grad=False)
        t_match = sample_t(N_M, requires_grad=False)

        model = BLPINN_3Module().to(DEVICE)

        T_train, loss_list_raw = train_joint(model, x_phi, t_h, xi_m, t_m, xi_p, t_p, t_match)
        
        # 训练计时彻底结束后，执行统一的数据跨硬件迁移与提取
        loss_history = torch.stack(loss_list_raw).detach().cpu().numpy().astype(np.float64)
        e_loss = float(loss_history[-1])

        T_train_per_iter_ms = T_train * 1e3 / TOTAL_JOINT_ITERS
        T_train_per_iter_point_us = T_train * 1e6 / TOTAL_LOSS_POINT_STEPS

        print(f" > trained: epochs={TOTAL_JOINT_ITERS}, T_train={T_train:.2f}s, e_loss={e_loss:.3e}")
        np.save(f"1d_BLPINN_loss_history_seed{seed}.npy", loss_history)

        model.eval()
        for p in model.parameters(): p.requires_grad_(False)

        for mu in MU_LIST:
            data_mu = lhs_data[mu]
            x_eval, t_eval, true_np = data_mu["x_lhs"], data_mu["t_lhs"], data_mu["true_lhs_np"]
            n_test = data_mu["n_test"]

            if n_test != NUM_SAMPLES:
                raise RuntimeError(f"N_test mismatch for mu={mu}: expected {NUM_SAMPLES}, got {n_test}.")

            # GPU Warm-up 阶段
            with torch.no_grad():
                for _ in range(EVAL_WARMUP): reconstruct_bipinn_gpu(model, x_eval, t_eval, mu)
            
            if torch.cuda.is_available(): torch.cuda.synchronize()
            eval_start = time.perf_counter()
            with torch.no_grad():
                for _ in range(EVAL_REPEAT): reconstruct_bipinn_gpu(model, x_eval, t_eval, mu)
            if torch.cuda.is_available(): torch.cuda.synchronize()
            T_eval = (time.perf_counter() - eval_start) / EVAL_REPEAT

            # 单独计算推断解以评估精度，排除在 T_eval 之外
            with torch.no_grad():
                u_eval_tensor = reconstruct_bipinn_gpu(model, x_eval, t_eval, mu)
            u_pred_lhs = u_eval_tensor.detach().cpu().numpy().reshape(-1)
            e2, einf = compute_error(true_np, u_pred_lhs)

            T_total = T_train + T_eval

            metrics[mu].append({
                "Seed": seed,
                "N_test": n_test,
                "e_loss": e_loss,
                "e2": e2,
                "einf": einf,
                "T_train": T_train,
                "T_eval": T_eval,
                "T_total": T_total,
                "T_train_per_iter_ms": T_train_per_iter_ms,
                "T_train_per_iter_point_us": T_train_per_iter_point_us,
                "total_trained_steps": TOTAL_JOINT_ITERS,
                "total_point_steps": TOTAL_LOSS_POINT_STEPS,
                "final_residual_points": N_PHI + N_H + 2 * N_Q + 2 * N_M,
                "eval_warmup": EVAL_WARMUP,
                "eval_repeat": EVAL_REPEAT
            })

            print(f"    -> [mu={mu}] N_test={n_test}, T_eval={T_eval:.6e}s, e2={e2:.3e}, einf={einf:.3e}")

            mu_label = f"{mu:.0e}"
            df_lhs_pred = pd.DataFrame({"t": data_mu["t_lhs_np"].reshape(-1), "x": data_mu["x_lhs_np"].reshape(-1), "u": u_pred_lhs})
            df_lhs_pred.to_csv(f"1d_BLPINN_U0_predicted_LHS_mu{mu_label}_seed{seed}.csv", index=False)

            if SAVE_FULL_GRID: save_full_grid_prediction(model, mu, seed)

    # =============================================================================
    # Summary tables (Strictly mirroring DAE formatting structure)
    # =============================================================================
    print("\n" + "=" * 80)
    print("ALL SEEDS COMPLETED. GENERATING SUMMARY TABLES.")
    print("=" * 80 + "\n")

    for mu in MU_LIST:
        dfm = pd.DataFrame(metrics[mu])
        mu_label = f"{mu:.0e}"
        dfm.to_csv(f"1d_BLPINN_mu{mu_label}_Metrics_Summary.csv", index=False)

        cols = [
            "e_loss", "e2", "einf", "T_train", "T_eval", "T_total",
            "T_train_per_iter_ms", "T_train_per_iter_point_us",
            "total_trained_steps", "total_point_steps", "final_residual_points", "N_test"
        ]
        stats = {c: mean_std(dfm[c].values) for c in cols}

        print(f"### Results for 1D BL-PINN, mu={mu} [Mean \pm Sample Std] ###")
        print(f"N_test: {stats['N_test'][0]:.0f} \pm {stats['N_test'][1]:.0f}")
        print(f"e_loss: {stats['e_loss'][0]:.3e} \pm {stats['e_loss'][1]:.3e}")
        print(f"e_2: {stats['e2'][0]:.3e} \pm {stats['e2'][1]:.3e}")
        print(f"e_inf: {stats['einf'][0]:.3e} \pm {stats['einf'][1]:.3e}")
        print(f"T_train (s): {stats['T_train'][0]:.2f} \pm {stats['T_train'][1]:.2f}")
        print(f"T_eval (s): {stats['T_eval'][0]:.6e} \pm {stats['T_eval'][1]:.6e}")
        print(f"T_total (s): {stats['T_total'][0]:.2f} \pm {stats['T_total'][1]:.2f}")
        print(f"T_train/iter (ms): {stats['T_train_per_iter_ms'][0]:.4f} \pm {stats['T_train_per_iter_ms'][1]:.4f}")
        print(f"T_train/(iter*pt) (us): {stats['T_train_per_iter_point_us'][0]:.4f} \pm {stats['T_train_per_iter_point_us'][1]:.4f}")
        print(f"Optimization steps: {stats['total_trained_steps'][0]:.1f} \pm {stats['total_trained_steps'][1]:.1f}")
        print(f"Point-iterations: {stats['total_point_steps'][0]:.1f} \pm {stats['total_point_steps'][1]:.1f}")
        print(f"Final residual points: {stats['final_residual_points'][0]:.1f} \pm {stats['final_residual_points'][1]:.1f}")
        print()


Starting 1D BL-PINN benchmark with strict LHS-based T_eval and error

Device: cuda
LHS test points: N_test=5000, LHS seed=1234
T_eval warmup: 20, repeated timing: 200


--------------------------------------------------------------------------------
Running seed = 33
--------------------------------------------------------------------------------
 > trained: epochs=20000, T_train=554.59s, e_loss=2.097e-03
    -> [mu=0.01] N_test=5000, T_eval=1.349893e-03s, e2=4.676e-03, einf=1.047e+00
    -> [mu=0.001] N_test=5000, T_eval=1.456320e-03s, e2=1.549e-02, einf=7.549e+00
    -> [mu=0.0001] N_test=5000, T_eval=1.404129e-03s, e2=2.637e-02, einf=1.445e+01

--------------------------------------------------------------------------------
Running seed = 99
--------------------------------------------------------------------------------
 > trained: epochs=20000, T_train=562.83s, e_loss=1.301e-03
    -> [mu=0.01] N_test=5000, T_eval=1.374396e-03s, e2=4.568e-03, einf=8.006e-01
    -> [mu=0.001] N_te